# unit02 レッスン: pandas によるデータ操作

**このレッスンで作れるようになるもの**: 表形式のデータ(行=サンプル、列=項目)を、
`DataFrame` という1つのオブジェクトとして持ち回り、
「特定の列だけ抜く」「条件で行を絞る」「欠損(空欄)を埋める」「グループごとに集計する」を
ループを1つも書かずに処理する。

これは機械学習の**前処理そのもの**です。scikit-learn にデータを渡す前に、必ずこの整形をします。
中身は unit01 の NumPy 配列(ブールマスク・axis 集計)がそのまま再登場します。

- 所要時間: 15〜25分
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
import pandas as pd
import numpy as np

def check(name, actual, expected, hint=""):
    try:
        ok = actual is not None and bool(np.all(np.isclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

# DataFrame / Series の中身を「値のリスト」にして比べる check 亜種。
# DataFrame同士は直接 == で比べると真偽が配列になって扱いにくいので、
# 値をフラットな list にそろえてから上の check に渡す。
def check_values(name, actual, expected, hint=""):
    try:
        vals = actual.values.ravel().tolist() if hasattr(actual, "values") else list(actual)
    except Exception:
        vals = actual
    return check(name, vals, expected, hint)

print("準備OK! pandas のバージョン:", pd.__version__)

---
## 概念1: DataFrame の構築と列選択 — 表データを1つのオブジェクトにする

### なぜ学ぶか
実務のデータは「顧客ID・年齢・購入額…」のような**表**です。C# なら `List<匿名型>` や
`List<Dictionary<string,object>>` で持ち回っていたものを、pandas では **1つの `DataFrame`** にまとめます。
CSV を読む・DB から取る・API の結果を整形する — データを扱う仕事はまず「表をDataFrameにする」から始まります。
そのあと「年齢の列だけ抜いて平均を見る」といった列単位の操作を延々とやります。

### 解説

**DataFrame** は「2次元の表」を表すオブジェクトです。C# のイメージだと **`List<匿名型>` + 列名つき**。
一番簡単な作り方は **列名 → 値のリスト の辞書**を `pd.DataFrame(...)` に渡すことです
(`pd.DataFrame` は「表を作るコンストラクタ」)。

```
C#:  var people = new[] { new {Name="A", Age=20}, new {Name="B", Age=30} };  // 行が主役
pandas:  pd.DataFrame({"name": ["A","B"], "age": [20,30]})                    # 列が主役(辞書のキー=列名)
```

表から**1列だけ**取り出すと **`Series`**(1次元の列)になります。書き方は辞書のキーアクセスそっくりの
`df["列名"]`。Series には `.mean()` や `.sum()`(LINQ の `Average()`/`Sum()` 相当)が生えています。
行数・列数は NumPy と同じ `df.shape`(`(行数, 列数)` のタプル)で分かります。

次のセルで、辞書からDataFrameを作り、列を1本抜く様子を見ます。

In [ ]:
# GOAL: 辞書から DataFrame を作り、1列(Series)を取り出して集計できることを確認する

# STEP 1: 列名 -> 値のリスト の辞書から表を作る(辞書のキーがそのまま列名になる)
df = pd.DataFrame({
    "name": ["Ann", "Bob", "Cho", "Dan"],
    "age":  [25, 32, 41, 28],
    "score": [80, 55, 90, 70],
})
print("=== DataFrame 全体 ===")
print(df)

# STEP 2: shape は (行数, 列数) のタプル — NumPy 配列と同じ感覚
print("\nshape:", df.shape)   # (4, 3) = 4行3列

# STEP 3: 1列を取り出すと Series(1次元の列)。辞書のキーアクセスに似た df["列名"]
ages = df["age"]
print("\n=== age 列(Series)===")
print(ages)

# STEP 4: Series には集計メソッドが生えている(LINQ の Average()/Max() 相当)
print("\n年齢の平均:", df["age"].mean(), " スコアの最大:", df["score"].max())

### 予測してみよう

次のセルは `df["score"]`(得点の列)に対して `.sum()`(合計)と `.mean()`(平均)を計算します。

**実行する前に**予測してください: スコアは `[80, 55, 90, 70]` です。合計はいくつ? 平均はいくつ?

In [ ]:
# 予測してから実行!
print("score の合計:", df["score"].sum())
print("score の平均:", df["score"].mean())

合計は `295`、平均は `73.75` です。列を取り出す → 集計、の2手だけ。C# の `list.Sum(x => x.Score)` が
`df["score"].sum()` に化けただけ、と捉えてください。

### 書いてみる

**課題**: 下のセルの `df` から **`age` 列を取り出し**、その値を `result1` に入れてください
(期待値: `[25, 32, 41, 28]`)。

ヒント(概念レベル): 辞書のキーアクセスに似た書き方で1列だけ抜く。1行で書けます。

In [ ]:
result1 = None
# ここに書く(result1 に df の age 列を代入する)


check_values("概念1: 列選択", result1, [25, 32, 41, 28],
             hint='1列取り出すのは df["列名"]。列名は "age"')

---
## 概念2: 行フィルタと loc / iloc — 条件で行を絞る・位置とラベルで取り出す

### なぜ学ぶか
「30歳以上の顧客だけ」「不良品フラグが立った行だけ」— 分析は**行の絞り込み**の連続です。
C# の LINQ `.Where(...)` に当たりますが、pandas では unit01 のブールマスクと**同じ発想**で書きます。
さらに、特定のセルや範囲を「ラベルで」または「位置で」取り出す `loc` / `iloc` は、
前処理コードのいたるところに出てきます。ここが読めないと他人のデータ処理が読めません。

### 解説

**行フィルタ**は2段構え(unit01 のブールマスクと完全に同じ発想):

```
C#:      people.Where(p => p.Age >= 30)
pandas:  df[df["age"] >= 30]      # df["age"] >= 30 が True/False の列(マスク)を作り、それで行を絞る
```

**loc と iloc** は「1マスや範囲を取り出す」ためのアクセサです。違いは指定方法だけ:

- **`df.loc[行ラベル, "列名"]`** … **ラベル(名前)**で指定。`df.loc[2, "age"]` = 行ラベル2・age列の値
- **`df.iloc[行番号, 列番号]`** … **位置(0始まりの整数)**で指定。`df.iloc[0]` = 先頭行、`df.iloc[:3]` = 先頭3行

> loc = **l**abel(ラベル)、iloc = **i**nteger(整数位置)、と頭文字で覚えると混同しません。

**複数列**を選ぶときは、列名の**リスト**を渡します: `df[["name", "age"]]`(角カッコが2重なのに注意)。
1列 `df["age"]` は Series、複数列 `df[["name","age"]]` は DataFrame が返ります。

次のセルで、フィルタと loc/iloc の動きを目で確認します。

In [ ]:
# GOAL: ブールマスクでの行フィルタと、loc / iloc の違いを目で確認する
df = pd.DataFrame({
    "name": ["Ann", "Bob", "Cho", "Dan"],
    "age":  [25, 32, 41, 28],
    "score": [80, 55, 90, 70],
})

# STEP 1: 条件式は True/False の列(マスク)を作る — これが行フィルタの正体
mask = df["age"] >= 30
print("=== マスク(bool の Series)===")
print(mask)

# STEP 2: マスクを df[...] に入れると True の行だけ残る(LINQ の Where 相当)
print("\n=== 30歳以上の行だけ ===")
print(df[df["age"] >= 30])

# STEP 3: iloc は「位置」で取り出す — 先頭2行(0,1行目)
print("\n=== iloc[:2] 先頭2行(位置指定)===")
print(df.iloc[:2])

# STEP 4: loc は「ラベル」で取り出す — 行ラベル2・age列の1マス
print("\nloc[2, 'age'] =", df.loc[2, "age"], "  ← 位置ではなくラベル2の行")

# STEP 5: 複数列は「列名のリスト」を渡す(角カッコ2重)
print("\n=== name と score の2列だけ ===")
print(df[["name", "score"]])

### 予測してみよう

次のセルは `df[df["score"] >= 75]`(得点75以上の行)を出します。scoreは `[80, 55, 90, 70]` です。

**実行前に予測**: 残るのは誰の行? 何行になりますか?(75ちょうどの人はいない点にも注意)

In [ ]:
# 予測してから実行!
high = df[df["score"] >= 75]
print(high)
print("\n残った行数:", len(high))

残るのは Ann(80)と Cho(90)の2行。`>= 75` なので境界の扱いを毎回確認する癖をつけましょう。

### 書いてみる

**課題**: `df` から **`age` が 30 以上の行**だけを絞り込み、そのうち **`name` 列**を取り出して
`result2` に入れてください(期待値: `["Bob", "Cho"]`)。

ヒント(概念レベル): まず `df[マスク]` で行を絞り、その結果に対して `["name"]` で列を取る。

In [ ]:
result2 = None
# ここに書く(result2 に「age30以上の行の name 列」を代入する)


# 文字列の Series なので値をリスト化して比べる
_r2 = result2.tolist() if hasattr(result2, "tolist") else result2
check("概念2: 行フィルタ+列選択", _r2, ["Bob", "Cho"],
      hint='df[df["age"] >= 30] で行を絞ってから ["name"] を付ける')

---
## 概念3: 欠損処理 — 空欄(NaN)を検出して埋める・捨てる

### なぜ学ぶか
現実のデータには**空欄**がつきものです(アンケート未回答、センサー欠測、結合で埋まらなかった列)。
pandas はこれを **`NaN`(Not a Number)** という特別な値で表します。C# の `null` に近いですが、
数値列の中に混ざる点が独特です。scikit-learn は NaN があると学習でエラーになるので、
**前処理で必ず「埋める(補完)」か「捨てる(除去)」の判断**をします。ここは unit05 の前処理で必ず再登場します。

### 解説

**欠損の検出**: `df["列"].isna()` は「各要素が NaN かどうか」の True/False の列を返します
(`is NaN` の略)。True の個数を数えれば欠損数が分かります(`.sum()` で True=1として合計 — unit01と同じ)。

**埋める**: `df["列"].fillna(値)` は NaN を指定した値に置き換えます。よく使うのは**その列の平均**で埋める形:

```python
df["age"].fillna(df["age"].mean())   # 平均で穴埋め(mean は NaN を自動で無視して計算してくれる)
```

**捨てる**: `df.dropna(subset=["列名"])` は指定列に NaN がある**行を丸ごと削除**します。

> `fillna` も `dropna` も**新しいDataFrameを返す**(元は変えない)。元を書き換えたいなら代入し直します。
> 前処理では「元データは残す」のが安全なので、この非破壊の挙動はむしろ好都合です。

次のセルで、わざと欠損を混ぜた表を作って処理します。

In [ ]:
# GOAL: NaN の検出・平均での穴埋め・行削除を目で確認する
# np.nan がその列の「空欄」。数値列に混ぜられる
df = pd.DataFrame({
    "name": ["Ann", "Bob", "Cho", "Dan"],
    "age":  [25, np.nan, 41, np.nan],   # Bob と Dan の年齢が欠損
    "score": [80, 55, np.nan, 70],       # Cho のスコアが欠損
})
print("=== 欠損を含む表 ===")
print(df)

# STEP 1: isna() で「どこが NaN か」を True/False で見る
print("\n=== age 列の isna() ===")
print(df["age"].isna())
print("age 列の欠損数:", df["age"].isna().sum())   # True の個数 = 欠損数

# STEP 2: fillna で NaN を平均で埋める(mean は NaN を無視して計算 → ここでは (25+41)/2 = 33)
print("\nage の平均(NaN無視):", df["age"].mean())
filled = df.copy()
filled["age"] = filled["age"].fillna(df["age"].mean())
print("=== age を平均で穴埋め ===")
print(filled)

# STEP 3: dropna で「score が欠損している行」を丸ごと削除
print("\n=== score が欠損の行を削除 ===")
print(df.dropna(subset=["score"]))

### 予測してみよう

上の `df` で、**`age` 列の欠損数**と、**`score` 列の欠損数**はそれぞれ何個でしょう?
表(Bob と Dan の age が空、Cho の score が空)を見て予測してから、次のセルを実行してください。

In [ ]:
# 予測してから実行!
print("age の欠損数  :", df["age"].isna().sum())
print("score の欠損数:", df["score"].isna().sum())

age は2個(Bob, Dan)、score は1個(Cho)。`isna().sum()` = 欠損数、が定番の数え方です。

### 書いてみる

**課題**: 上の `df` の **`score` 列の欠損を、その列の平均で埋めた**とき、
**Cho の score(=平均値)**がいくつになるかを `result3` に入れてください。
scoreは `[80, 55, NaN, 70]` なので、平均は NaN を無視した `(80+55+70)/3` です(期待値: `68.333...`)。

ヒント(概念レベル): 平均は `df["score"].mean()`(NaN は自動で無視される)。その値がそのまま穴を埋める値。

In [ ]:
result3 = None
# ここに書く(result3 に「score列の平均 = 埋められる値」を代入する)


check("概念3: 欠損処理(平均補完)", result3, 68.33333333,
      hint='df["score"].mean() が平均(NaNは無視)。それが穴埋めに使う値')

---
## 概念4: groupby 集計 — グループごとにまとめて集計する

### なぜ学ぶか
「部署ごとの平均給与」「商品カテゴリごとの売上合計」「あやめの品種ごとの平均花弁長」—
**カテゴリで束ねて集計**するのは分析の花形です。C# の LINQ `GroupBy` そのもの。
機械学習でも「クラスごとに特徴量の平均を見る」といった探索でよく使い、
unit04 の分類データを眺めるときにも効きます。

### 解説

**`df.groupby("列A")["列B"].mean()`** は「列Aの値ごとに列Bの平均を出す」です。C#と並べると:

```
C#:      list.GroupBy(x => x.Dept).Select(g => new { g.Key, Avg = g.Average(x => x.Salary) })
pandas:  df.groupby("dept")["salary"].mean()
```

戻り値は **Series**(index=グループの値、value=集計結果)。`.mean()` の代わりに
`.sum()` / `.count()` / `.max()` も使えます。

**複数の集計を同時に**出したいときは **`.agg([...])`**(aggregate=集計)にリストを渡します:

```python
df.groupby("dept")["salary"].agg(["mean", "count"])   # 平均と件数を同時に → 列名 mean / count の表
```

これは LINQ で `g => new { Avg=..., Count=... }` と匿名型に複数プロパティを詰めるのと同じ発想です。
次のセルで、カテゴリ列で束ねて集計します。

In [ ]:
# GOAL: groupby で「グループごとの平均」と「複数集計の同時算出」を確認する
df = pd.DataFrame({
    "dept":   ["sales", "sales", "dev", "dev", "dev"],
    "salary": [300, 340, 500, 460, 520],
})
print("=== 元データ ===")
print(df)

# STEP 1: dept ごとに salary の平均(戻り値は index=dept の Series)
print("\n=== 部署ごとの平均給与 ===")
print(df.groupby("dept")["salary"].mean())

# STEP 2: グループごとの件数(何行あるか)
print("\n=== 部署ごとの人数 ===")
print(df.groupby("dept")["salary"].count())

# STEP 3: agg に集計名のリストを渡すと複数を同時に(列名 mean / count の表になる)
print("\n=== 平均と人数を同時に ===")
print(df.groupby("dept")["salary"].agg(["mean", "count"]))

### 予測してみよう

上の `df` で `df.groupby("dept")["salary"].mean()` を出すと、**sales と dev の平均**はそれぞれいくつ?
salesは `[300, 340]`、devは `[500, 460, 520]` です。予測してから次を実行してください。

In [ ]:
# 予測してから実行!
means = df.groupby("dept")["salary"].mean()
print(means)
print("\ndev の平均だけ取り出す:", means["dev"])

sales は `(300+340)/2 = 320`、dev は `(500+460+520)/3 = 493.33...`。
groupby の結果は Series なので `means["dev"]` のようにグループ名で1つだけ取り出せます。

### 書いてみる

**課題**: 上の `df` を **`dept` ごとに `salary` の平均**で集計し、その結果(2グループ分の平均)を
`result4` に入れてください。並び順は index 順(dev, sales のアルファベット順)になります
(期待値: `[493.33..., 320.0]` = dev, sales の順)。

ヒント(概念レベル): `df.groupby("列")["列"].mean()`。groupby の結果は index がアルファベット順にそろう。

In [ ]:
result4 = None
# ここに書く(result4 に「deptごとのsalary平均」の Series を代入する)


check_values("概念4: groupby集計", result4, [493.33333333, 320.0],
             hint='df.groupby("dept")["salary"].mean()。結果は dev, sales の順に並ぶ')

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | C#で言うと |
|------|--------|-----------|
| DataFrame構築・列選択 | 辞書から表を作り `df["列"]` で1列(Series)を抜く | `List<匿名型>` + 列名、`Select(x=>x.Prop)` |
| 行フィルタ / loc・iloc | `df[df["列"]>x]` で絞る、loc=ラベル・iloc=位置 | `Where(...)`、インデクサ |
| 欠損処理 | `isna` で検出、`fillna`/`dropna` で埋める・捨てる | `null` チェックと除去 |
| groupby集計 | `groupby("A")["B"].mean()`、`.agg([...])` で複数同時 | `GroupBy().Select(g=>new{...})` |

**この先どこで使うか**: unit03 では、ここで整えた表を `train_test_split` に渡して回帰モデルを学習します
(DataFrame のまま fit できます)。unit05 の前処理では今日の**欠損補完**が実戦の形で再登場します。
行フィルタのブールマスクは unit01 の NumPy マスクと同じ発想 — 土台はぜんぶ地続きです。

**次**: 演習 `ex01_dataframe.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/ml-intro/unit02-pandas/tests/test_ex01.py -q`